# 04 - Segmentation Analysis

This notebook segments customers for targeted retention campaigns:

1. **RFM Segmentation**: Classic marketing segmentation
2. **Behavioral Clustering**: Data-driven segments
3. **Segment Profiling**: Understand each segment's characteristics
4. **Targeting Strategy**: Prioritize segments for intervention

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.segmentation import CustomerSegmenter

# Settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load features and predictions
features = pd.read_parquet('../data/features/churn_features.parquet')
predictions = pd.read_parquet('../data/features/churn_predictions.parquet')

# Merge
df = features.merge(predictions[['visitorid', 'churn_proba']], on='visitorid')
print(f"Loaded {len(df):,} customers")

## 2. RFM Segmentation

RFM (Recency, Frequency, Monetary) is a classic marketing segmentation method.

In [ ]:
# Initialize segmenter
segmenter = CustomerSegmenter()

In [ ]:
# Compute RFM scores and assign segments
rfm = segmenter.rfm_segment(
    features=df,
    recency_col='days_since_last_purchase',
    frequency_col='transaction_count',
    monetary_col='total_items_purchased'
)

print("RFM Scores Sample:")
rfm.head(10)

In [ ]:
# Segment distribution
segment_dist = rfm['segment'].value_counts()
print("RFM Segment Distribution:")
print(segment_dist)

In [ ]:
# Visualize segment distribution
fig, ax = plt.subplots(figsize=(12, 6))
segment_dist.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Customers')
ax.set_title('RFM Segment Distribution')

# Add percentage labels
for i, (segment, count) in enumerate(segment_dist.sort_values().items()):
    pct = count / len(rfm) * 100
    ax.text(count + 10, i, f'{pct:.1f}%', va='center')

plt.tight_layout()
plt.savefig('../figures/rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Merge segments with features and predictions
df_segmented = df.merge(rfm[['visitorid', 'segment', 'R_score', 'F_score', 'M_score']], on='visitorid')
print(f"Segmented dataset: {len(df_segmented):,} rows")

## 3. Segment Profiles

In [ ]:
# Get segment profiles
profiles = segmenter.get_segment_profiles(
    features=df,
    segments=rfm,
    churn_proba=df_segmented['churn_proba'],
    segment_col='segment'
)

# Display as table
profile_table = segmenter.get_segment_summary_table(profiles)
profile_table

In [ ]:
# Detailed profile for each segment
for profile in profiles[:5]:  # Top 5 segments
    print(f"\n{'='*60}")
    print(f"SEGMENT: {profile.name}")
    print(f"{'='*60}")
    print(f"Size: {profile.size:,} customers ({profile.size_pct:.1%})")
    print(f"Avg Churn Risk: {profile.avg_churn_prob:.1%}" if profile.avg_churn_prob else "N/A")
    print(f"\nDescription: {profile.description}")
    print(f"\nRecommended Action: {profile.recommended_action}")

In [ ]:
# Churn risk by segment
churn_by_segment = df_segmented.groupby('segment')['churn_proba'].agg(['mean', 'count'])
churn_by_segment.columns = ['Avg Churn Risk', 'Count']
churn_by_segment = churn_by_segment.sort_values('Avg Churn Risk', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(churn_by_segment)))
churn_by_segment['Avg Churn Risk'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Average Churn Probability')
ax.set_title('Churn Risk by RFM Segment')
ax.axvline(x=df_segmented['churn_proba'].mean(), color='red', linestyle='--', label='Overall Average')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/churn_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Behavioral Clustering

In [ ]:
# Find optimal number of clusters
cluster_features = ['days_since_last_purchase', 'transaction_count', 'total_events',
                   'view_to_purchase_rate', 'activity_trend', 'session_count']
cluster_features = [f for f in cluster_features if f in df.columns]

inertias = segmenter.find_optimal_clusters(df, cluster_features, max_clusters=10)

# Plot elbow curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(inertias.keys()), list(inertias.values()), 'bo-')
ax.set_xlabel('Number of Clusters')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method for Optimal K')
plt.tight_layout()
plt.show()

In [ ]:
# Perform clustering with chosen K
n_clusters = 5  # Choose based on elbow
clusters = segmenter.behavioral_cluster(df, cluster_features, n_clusters=n_clusters)

print(f"Cluster distribution:")
print(clusters['cluster'].value_counts().sort_index())

In [ ]:
# Add clusters to data
df_clustered = df.merge(clusters, on='visitorid')

# Profile clusters
cluster_profiles = df_clustered.groupby('cluster')[cluster_features + ['churn_proba']].mean()
print("Cluster Profiles:")
cluster_profiles

In [ ]:
# Radar chart for cluster profiles
from math import pi

# Normalize features for radar chart
radar_features = cluster_features[:5]  # Limit to 5 features
normalized = cluster_profiles[radar_features].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))

angles = [n / float(len(radar_features)) * 2 * pi for n in range(len(radar_features))]
angles += angles[:1]  # Close the polygon

colors = plt.cm.Set2(range(n_clusters))

for i, (cluster_id, row) in enumerate(normalized.iterrows()):
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}', color=colors[i])
    ax.fill(angles, values, alpha=0.15, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features)
ax.set_title('Cluster Profiles (Radar Chart)')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
plt.tight_layout()
plt.savefig('../figures/cluster_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. High-Value At-Risk Customers

In [ ]:
# Identify high-value at-risk customers
high_value_at_risk = segmenter.target_high_value_at_risk(
    features=df,
    segments=rfm,
    churn_proba=df['churn_proba'],
    risk_threshold=0.5,
    value_percentile=0.75
)

print(f"High-Value At-Risk Customers: {len(high_value_at_risk):,}")
print(f"Percentage of base: {len(high_value_at_risk)/len(df)*100:.1f}%")
high_value_at_risk.head(10)

In [ ]:
# Value-Risk quadrant analysis
value_col = 'total_items_purchased' if 'total_items_purchased' in df.columns else 'transaction_count'

fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot
scatter = ax.scatter(
    df[value_col],
    df['churn_proba'],
    c=df['churned'],
    cmap='RdYlGn_r',
    alpha=0.5,
    s=20
)

# Add quadrant lines
value_median = df[value_col].median()
risk_median = df['churn_proba'].median()
ax.axhline(y=risk_median, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=value_median, color='gray', linestyle='--', alpha=0.5)

# Label quadrants
ax.text(df[value_col].max()*0.8, 0.9, 'High Value\nHigh Risk', ha='center', fontsize=10, color='red')
ax.text(df[value_col].min()+1, 0.9, 'Low Value\nHigh Risk', ha='center', fontsize=10, color='orange')
ax.text(df[value_col].max()*0.8, 0.1, 'High Value\nLow Risk', ha='center', fontsize=10, color='green')
ax.text(df[value_col].min()+1, 0.1, 'Low Value\nLow Risk', ha='center', fontsize=10, color='gray')

ax.set_xlabel('Customer Value (Total Items Purchased)')
ax.set_ylabel('Churn Probability')
ax.set_title('Value-Risk Quadrant Analysis')
plt.colorbar(scatter, label='Churned')
plt.tight_layout()
plt.savefig('../figures/value_risk_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Targeting Strategy Summary

In [ ]:
# Priority targeting matrix
priority_segments = [
    'Cannot Lose Them',  # High value, high risk
    'At Risk',           # Good customers slipping away
    'About to Sleep',    # Early intervention possible
]

priority_customers = df_segmented[df_segmented['segment'].isin(priority_segments)]

print("Priority Targeting Summary:")
print(f"{'='*60}")
for segment in priority_segments:
    seg_data = priority_customers[priority_customers['segment'] == segment]
    if len(seg_data) > 0:
        print(f"\n{segment}:")
        print(f"  Customers: {len(seg_data):,}")
        print(f"  Avg Churn Risk: {seg_data['churn_proba'].mean():.1%}")
        print(f"  Action: {segmenter.SEGMENT_ACTIONS.get(segment, 'Custom campaign')}")

In [ ]:
# Save segmented data
df_segmented.to_parquet('../data/features/segmented_customers.parquet', index=False)
print(f"\nSaved segmented data for {len(df_segmented):,} customers")

## 7. Key Insights

### RFM Segments
- Clear differentiation between loyal and at-risk customers
- "Cannot Lose Them" and "At Risk" segments are priority targets
- Champions show lowest churn risk - protect these relationships

### Behavioral Clusters
- Data-driven clusters reveal hidden patterns
- Engagement patterns differ significantly across clusters
- Useful for personalization beyond RFM

### Targeting Recommendations
1. **Immediate**: Contact "Cannot Lose Them" with personalized offers
2. **Urgent**: Re-engage "At Risk" with value reminders
3. **Proactive**: Nurture "About to Sleep" before they disengage
4. **Maintain**: Keep "Champions" happy with rewards